# Streaming the ICP parameter API

This notebook drives the whole pipeline end to end and then watches it produce estimates:

1. validate the API that is on disk (`serve_api.py --check`)
2. start the HTTP server
3. read the **contract** — the six parameters, their bounds, and their provenance tags
4. start an estimator — the batch refit (`model/stream_refit.jl`) or the sequential filter
   (`model/stream_filter.jl`) — which appends one JSON record per estimate and flushes
5. poll `/latest/<patient>` live while that runs
6. poll `/stream/<patient>/<id>?since=<t>` incrementally — only what is new
7. plot the confidence intervals collapsing as the compression manoeuvre supplies the
   information

**Read the intervals, not the values.** Before the manoeuvre, the venous-path parameters are
not estimated, they are guessed; every record carries an interval and an `identifiable`
flag, and `status` is `unidentifiable` whenever the flag is false regardless of where the
point estimate falls. Watching those intervals collapse is the intended use of this API.
The two estimators compute that interval differently — Fisher information at a converged
optimum, versus a filter covariance — which step 4 lays out.

---

### Environment

The server needs nothing but the standard library. This notebook additionally wants
`matplotlib` for the plots (every cell degrades gracefully without it):

```bash
python3 -m venv .venv-dash
.venv-dash/bin/pip install jupyterlab matplotlib
.venv-dash/bin/jupyter lab notebooks/stream_explorer.ipynb
```

Step 4 additionally needs `julia` on PATH and the recordings in `data/` (gitignored — see
`scripts/setup_data.sh`). Steps 1–3 and 5–7 work against whatever streams already exist, so
you can skip step 4 entirely and explore the committed data.

## 0. Configuration

`PORT` defaults to 8770. Anything below 1024 is a privileged port and will fail with
`PermissionError: [Errno 13]` unless you are root — that is an OS rule, not a bug in the
server.

In [1]:
import json, math, os, subprocess, sys, time
import urllib.request, urllib.error
from pathlib import Path


def find_root(start=None):
    """Walk up from the notebook until we find the repo (api/index.json is the marker)."""
    p = Path(start or os.getcwd()).resolve()
    for c in (p, *p.parents):
        if (c / "api" / "index.json").is_file():
            return c
    raise SystemExit("could not find the repo root — no api/index.json above " + str(p))


ROOT    = find_root()
PY      = sys.executable
PORT    = int(os.environ.get("ICP_API_PORT", 8770))
BASE    = f"http://127.0.0.1:{PORT}"
PATIENT = "081"                      # one of the four in api/index.json

INDEX  = json.loads((ROOT / "api" / "index.json").read_text())
PARAMS = INDEX["params"]

# The two child processes this notebook may own. Declared here so that every cell below can
# be run in any order, whether or not step 2 and step 4 have been run.
SERVER = globals().get("SERVER")      # the HTTP server   (step 2)
REFIT  = globals().get("REFIT")       # the julia refit   (step 4)


def stream_writer(patient=PATIENT):
    """pid of whatever is currently writing this patient's stream, or None.

    model/stream_contract.jl takes this lock at startup and releases it at exit, so it is
    authoritative no matter who started the run — this notebook, a terminal, or cron. That
    matters: the live cells below key off it, and an estimator you started in a terminal is
    the normal case, not the exception."""
    lock = ROOT / "api" / "stream" / patient / ".writer.pid"
    try:
        pid = int(lock.read_text().strip())
    except (OSError, ValueError):
        return None
    try:
        os.kill(pid, 0)                   # signal 0: liveness probe, sends nothing
    except ProcessLookupError:
        return None                       # stale lock, holder is gone
    except PermissionError:
        pass                              # alive, just not ours
    return pid


def refit_alive():
    """True while ANY estimator is writing this patient's stream."""
    r = globals().get("REFIT")
    return (r is not None and r.poll() is None) or stream_writer() is not None


print("root      ", ROOT)
print("python    ", PY)
print("julia     ", subprocess.run(["which", "julia"], capture_output=True, text=True).stdout.strip() or "NOT ON PATH — step 4 will be skipped")
print("data/     ", "present" if (ROOT / "data" / "jugular_prepared").is_dir() else "MISSING — run scripts/setup_data.sh (step 4 needs it)")
print("patients  ", ", ".join(INDEX["patients"]))
print("params    ", "\n           ".join(PARAMS))

root       /home/thermistor/Documents/icp-parameter-api
python     /home/thermistor/Documents/brainphysio_celia/.venv-dash/bin/python
julia      /home/thermistor/.juliaup/bin/julia
data/      present
patients   081, 083, 137, 150
params     craniospinal_compliance
           csf_outflow_resistance
           arterial_compliance
           collateral_outflow_resistance
           jugular_resting_area
           sagittal_sinus_pressure


## 1. Validate the API on disk

`--check` never serves. It asserts that every parameter has metadata, that the four bounds
are strictly ordered (`min_path < min_healthy < max_healthy < max_path`), and reports how
many records each patient's stream currently holds. Exit code 1 means the API on disk is not
coherent.

In [2]:
chk = subprocess.run([PY, str(ROOT / "scripts" / "serve_api.py"), "--check"],
                     cwd=ROOT, capture_output=True, text=True)
print(chk.stdout or chk.stderr)
print("exit:", chk.returncode, "— OK" if chk.returncode == 0 else "— API on disk is not coherent")

index: 6 params, 4 patients

  craniospinal_compliance          [0.1, 0.5, 1.1, 2.0]         [clinical]
  csf_outflow_resistance           [3.0, 6.0, 10.0, 18.0]       [clinical]
  arterial_compliance              [0.005, 0.015, 0.05, 0.1]    [model]
  collateral_outflow_resistance    [0.01, 0.02, 0.2, 2.0]       [model]
  jugular_resting_area             [0.3, 0.8, 2.0, 4.0]         [qc]
  sagittal_sinus_pressure          [2.0, 5.0, 8.0, 15.0]        [clinical]

  patient 081: 342 records
  patient 083: 0 records   (no stream yet — run model/stream_refit.jl)
  patient 137: 0 records   (no stream yet — run model/stream_refit.jl)
  patient 150: 0 records   (no stream yet — run model/stream_refit.jl)

exit: 0 — OK


## 2. Start the server

The API is **file-backed**: each request re-reads the `.jsonl` on disk, so a refit that is
still running is served live. The server is started as a child process of this kernel; the
last cell of the notebook shuts it down.

In [3]:
def api(path, timeout=10):
    """GET a JSON route. Raises urllib.error.HTTPError on 404/500 (the body is still JSON)."""
    with urllib.request.urlopen(BASE + path, timeout=timeout) as r:
        return json.load(r)


def server_up():
    try:
        api("/", timeout=1)
        return True
    except Exception:
        return False


if server_up():
    print(f"already serving at {BASE}  (not starting a second one)")
else:
    SERVER = subprocess.Popen([PY, str(ROOT / "scripts" / "serve_api.py"), "--port", str(PORT)],
                              cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(60):
        if server_up():
            break
        if SERVER.poll() is not None:                      # died on startup
            raise SystemExit("server exited:\n" + SERVER.stdout.read())
        time.sleep(0.1)
    else:
        raise SystemExit(f"server did not come up on {BASE}")
    print(f"serving at {BASE}  (pid {SERVER.pid})")

print()
print(json.dumps(api("/"), indent=1))

already serving at http://127.0.0.1:8770  (not starting a second one)

{
 "params": [
  "craniospinal_compliance",
  "csf_outflow_resistance",
  "arterial_compliance",
  "collateral_outflow_resistance",
  "jugular_resting_area",
  "sagittal_sinus_pressure"
 ],
 "patients": [
  "081",
  "083",
  "137",
  "150"
 ],
 "param_metadata": "params/{id}.json",
 "stream": "stream/{patient}/{id}.jsonl",
 "stream_record": "{t_end, value, ci_lo, ci_hi, status, identifiable, rmse, n_solves}",
 "status_values": [
  "healthy",
  "borderline",
  "pathological",
  "unidentifiable"
 ]
}


## 3. The contract: what each parameter means

`api/params/*.json` is committed — it is the contract, and it is generated by
`model/api_spec.jl`. **The provenance tag is the part to read.**

- `clinical` — a published human reference range for a directly measured quantity.
- `model` — no human reference range exists; the bounds are anchored on the source model's
  literature default, so they describe the *model*, not physiology. Do not show these to a
  clinician as normal ranges.
- `qc` — streamed as a check on the fit. `jugular_resting_area` is the only one: a value
  outside the band is evidence the **fit** is wrong, not that the patient is sick.

In [4]:
meta = api("/params")

hdr = f'{"parameter":<31}{"unit":<16}{"pathological / healthy / pathological":<40}{"prov":<10}concern'
print(hdr)
print("-" * len(hdr))
for pid in PARAMS:
    m = meta[pid]
    band = (f'{m["min_pathological_bound"]:g} < {m["min_healthy_bound"]:g} .. '
            f'{m["max_healthy_bound"]:g} < {m["max_pathological_bound"]:g}')
    print(f'{pid:<31}{m["unit"]:<16}{band:<40}{m["bound_provenance"]:<10}{m["direction_of_concern"]}')

print()
one = api(f"/params/{PARAMS[0]}")
print(one["id"], "—", one["physiological_equivalent_description"])
print()
print("bound source:", one["bound_source"])
print("caveat      :", one["caveat"])

parameter                      unit            pathological / healthy / pathological   prov      concern
--------------------------------------------------------------------------------------------------------
craniospinal_compliance        mL/mmHg         0.1 < 0.5 .. 1.1 < 2                    clinical  low
csf_outflow_resistance         mmHg/(mL/min)   3 < 6 .. 10 < 18                        clinical  both
arterial_compliance            mL/mmHg         0.005 < 0.015 .. 0.05 < 0.1             model     low
collateral_outflow_resistance  mmHg.s/mL       0.01 < 0.02 .. 0.2 < 2                  model     high
jugular_resting_area           cm2 per vein    0.3 < 0.8 .. 2 < 4                      qc        high
sagittal_sinus_pressure        mmHg            2 < 5 .. 8 < 15                         clinical  high

craniospinal_compliance — Craniospinal compliance: the volume the craniospinal space accepts per mmHg of ICP rise. The bedside quantity behind PVI, RAP and 'compensatory reserve'.

## 4. Start an estimator — the thing that produces the stream  ⚠️ optional, destructive

Two estimators write this stream. They are not interchangeable and the choice matters:

| | `"batch"` — `model/stream_refit.jl` | `"filter"` — `model/stream_filter.jl` |
|---|---|---|
| method | re-fits the whole recording at each `t_end` | sequential joint state+parameter filter |
| cost per record | ~55 full re-integrations (~2.5 min) | 39 one-second integrations (~0.2 s) |
| vs. real time | ~11x slower | ~4.5x **faster** |
| interval | Fisher information at a converged optimum | filter covariance, given assumed process noise |
| parameters | constant within a window | allowed to drift (a random walk you choose) |

The batch fit is the reference: its intervals are the ones the README's claims are about.
The filter is the one that can keep up with a patient. Where they disagree, the batch fit is
the thing to trust and the filter is the thing to deploy — that tension is the honest state
of it, not a bug to be tuned away.

`bash model/stream_all.sh 16 45` runs the batch fit on all four patients, ~70 min.

Two things to know before you set `START_REFIT = True`:

- **It truncates.** The streams are opened `"w"`, so starting a refit wipes this patient's
  existing `api/stream/<patient>/*.jsonl` and starts over at `t_end = −60`.
- It needs `julia` and the recordings in `data/`. If either is missing this cell no-ops and
  the rest of the notebook still works against the streams already on disk.

In [5]:
START_REFIT = True         # <- flip to True to actually run it
ESTIMATOR   = "filter"     # "filter" (~90 s for the recording) or "batch" (~1 h)
NWIN, ITERS = 4, 12        # batch only; small demo — real runs are 16 windows / 45 iterations

LOG = ROOT / "api" / f"refit_{PATIENT}.log"

have_julia = subprocess.run(["which", "julia"], capture_output=True).returncode == 0
have_data  = (ROOT / "data" / "jugular_prepared").is_dir()


ESTIMATORS = ("stream_refit.jl", "stream_filter.jl")   # batch fit, sequential filter


def other_refits(patient=PATIENT):
    """Estimators writing this patient's streams in this repo that this notebook did not start.

    This matters more than it looks: both estimators open the six .jsonl files with "w".
    Two of them running at once truncate and then overwrite each other at their own file
    offsets, so the stream becomes torn records rather than data. One writer per patient —
    and that includes one of each, which is the easy mistake to make.
    (Linux only — /proc. Returns [] elsewhere.)"""
    mine = REFIT.pid if REFIT is not None else None
    found = []
    try:
        out = subprocess.run(["pgrep", "-f", "|".join(ESTIMATORS)],
                             capture_output=True, text=True).stdout.split()
    except FileNotFoundError:
        return []
    for pid in (int(x) for x in out):
        if pid == mine:
            continue
        try:
            if Path(f"/proc/{pid}/cwd").resolve() != ROOT:
                continue                      # a refit in some other checkout
            env = dict(kv.split("=", 1) for kv in
                       Path(f"/proc/{pid}/environ").read_bytes().decode().split("\0")
                       if "=" in kv)
        except (OSError, ValueError):
            continue
        if env.get("_PATIENT", "081") == patient:
            found.append(pid)
    return found


if not START_REFIT:
    print("START_REFIT is False — skipping. The notebook will read whatever is already in "
          f"api/stream/{PATIENT}/")
elif not (have_julia and have_data):
    print("skipping:", "julia not on PATH. " if not have_julia else "",
          "data/jugular_prepared missing (run scripts/setup_data.sh). " if not have_data else "")
elif REFIT is not None and REFIT.poll() is None:
    # this notebook's own child — the only case where REFIT.pid exists
    print(f"this notebook already started one (pid {REFIT.pid}) — not starting a second")
elif stream_writer() is not None or other_refits():
    # anyone else's: a terminal, cron, another notebook. refit_alive() covers both cases,
    # so it must NOT be the test here — it is true when REFIT is None.
    who = stream_writer() or other_refits()
    print(f"NOT starting: something else is already writing patient {PATIENT}'s stream "
          f"(pid {who}).\n"
          f"Both estimators open the same six .jsonl files with \"w\" and overwrite each\n"
          f"other. model/stream_contract.jl refuses this at startup anyway — this is the\n"
          f"same check, one step earlier and with a better message.\n"
          f"Stop that one first, or set PATIENT to a patient nobody else is fitting.")
else:
    env = {**os.environ, "_PATIENT": PATIENT, "_NWIN": str(NWIN), "_WIN_ITERS": str(ITERS)}
    script = "model/stream_filter.jl" if ESTIMATOR == "filter" else "model/stream_refit.jl"
    LOGF  = LOG.open("w")
    REFIT = subprocess.Popen(["julia", "--project=.", script],
                             cwd=ROOT, env=env, stdout=LOGF, stderr=subprocess.STDOUT)
    print(f"{ESTIMATOR} running (pid {REFIT.pid}) — {script}")
    print("  one record per second of data, ~90 s for the whole recording"
          if ESTIMATOR == "filter" else
          f"  {NWIN} windows x {ITERS} iterations, ~2.5 min per window")
    print(f"log: {LOG}   —  the next cell tails it")

NOT starting: something else is already writing patient 081's stream (pid 279651).
Both estimators open the same six .jsonl files with "w" and overwrite each
other. model/stream_contract.jl refuses this at startup anyway — this is the
same check, one step earlier and with a better message.
Stop that one first, or set PATIENT to a patient nobody else is fitting.


In [6]:
def tail(path=LOG, n=25):
    """Last n lines of the refit log. Re-run this cell to follow progress."""
    if not Path(path).is_file():
        return f"(no log at {path} yet)"
    return "".join(Path(path).read_text(errors="replace").splitlines(keepends=True)[-n:])


print(f"refit: {'running' if refit_alive() else 'not running'}\n")
print(tail())

refit: running

   -87.0    -0.119    0.216       0    0.01212 ????      2.606 ????     0.1386 ????      5.783 ????      6.059 ????      7.183  ok 
   -86.0     0.030    0.213       0    0.01212 ????      2.606 ????     0.1387 ????      5.782 ????      6.059 ????      7.184  ok 
   -85.0     0.090    0.210       0    0.01212 ????      2.606 ????     0.1393 ????      5.779 ????      6.059 ????      7.186  ok 
   -84.0     0.056    0.208       0    0.01212 ????      2.606 ????     0.1395 ????      5.777 ????      6.059 ????      7.187  ok 
   -83.0    -0.062    0.205       0    0.01212 ????      2.606 ????     0.1397 ????      5.779 ????      6.059 ????      7.185  ok 
   -82.0     0.043    0.202       0    0.01212 ????      2.606 ????     0.1396 ????      5.777 ????      6.059 ????      7.187  ok 
   -81.0     0.118    0.201       0    0.01212 ????      2.606 ????     0.1403 ????      5.774 ????      6.059 ????      7.189  ok 
   -80.0     0.119    0.199       0    0.01212 ????      2.6

## 5. Live view of `/latest/<patient>`

`/latest` collapses every parameter to its most recent record and joins it against the
metadata. This loop re-reads it every few seconds — run it while the refit above is going
and you will see rows appear and then move.

Columns: the point estimate, its 95% interval, the healthy band from the contract, and the
verdict. `????` means `identifiable` is false — the interval is so wide the number is not a
measurement, and `status` is forced to `unidentifiable`.

In [ ]:
from IPython.display import clear_output


def fmt_latest(patient=PATIENT):
    try:
        d = api(f"/latest/{patient}")["params"]
    except urllib.error.URLError as e:
        return f"server unreachable: {e}"
    if not d:
        return (f"patient {patient}: no records yet.\n"
                f"  If no refit is running: start one (step 4), or pick a PATIENT that has a "
                f"stream.\n"
                f"  If one IS running: it writes nothing until its first window finishes — "
                f"~4 min at the\n  default 45 iterations, and the six .jsonl files sit at 0 "
                f"bytes until then. Same for the log.")

    out = [f'{"parameter":<31}{"value":>11}{"95% interval":>28}{"healthy":>20}   status',
           "-" * 100]
    for pid in PARAMS:
        r = d.get(pid)
        if r is None:
            out.append(f"{pid:<31}{'—':>11}")
            continue
        ci = ("—" if r["ci_lo"] is None
              else f'[{r["ci_lo"]:.3g}, {r["ci_hi"]:.3g}]')
        lo, hi = r["healthy_range"]
        flag = "" if r["identifiable"] else "  ????"
        out.append(f'{pid:<31}{r["value"]:>11.4g}{ci:>28}'
                   f'{f"[{lo:g}, {hi:g}]":>20}   {r["status"]}{flag}')
    n = max((r["n_records"] for r in d.values()), default=0)
    t = max((r["t_end"] for r in d.values() if r["t_end"] is not None), default=float("nan"))
    out.append("")
    out.append(f"{n} windows so far, latest t_end = {t:g} s")
    return "\n".join(out)


WATCH_SECONDS, INTERVAL = 900, 5     # only matters while an estimator is actually running

t0, seen_t, seen_at = time.time(), None, time.time()
while True:
    running = refit_alive()
    try:
        newest = max((r["t_end"] for r in api(f"/latest/{PATIENT}")["params"].values()
                      if r["t_end"] is not None), default=None)
    except urllib.error.URLError:
        newest = None
    if newest != seen_t:
        seen_t, seen_at = newest, time.time()
    clear_output(wait=True)
    print(f"{BASE}/latest/{PATIENT}      {time.strftime('%H:%M:%S')}\n")
    print(fmt_latest())
    who = stream_writer()
    print("\nestimator:", f"running (pid {who}, holds the stream lock)" if who
          else "running (started by this notebook)" if running else "not running")
    print(f"newest record {time.time() - seen_at:.0f}s ago"
          f"  — a paced run at _SPEED=0.02 emits one every 50s")
    if not running:                       # nothing will change — render once and return
        break
    if time.time() - t0 >= WATCH_SECONDS:
        print(f"(stopped watching after {WATCH_SECONDS}s — re-run this cell to keep going)")
        break
    time.sleep(INTERVAL)

http://127.0.0.1:8770/latest/081      17:06:19

parameter                            value                95% interval             healthy   status
----------------------------------------------------------------------------------------------------
craniospinal_compliance             0.1454             [0.0422, 0.501]          [0.5, 1.1]   unidentifiable  ????
csf_outflow_resistance               5.765                 [1.6, 20.8]             [6, 10]   unidentifiable  ????
arterial_compliance                 0.0121           [0.00306, 0.0478]       [0.015, 0.05]   unidentifiable  ????
collateral_outflow_resistance        2.606                [0.66, 10.3]         [0.02, 0.2]   unidentifiable  ????
jugular_resting_area                 6.058                [1.53, 23.9]            [0.8, 2]   unidentifiable  ????
sagittal_sinus_pressure              7.191                [4.23, 12.2]              [5, 8]   healthy

57 windows so far, latest t_end = -63 s

estimator: running (pid 279651, holds 

## 6. Incremental polling with `?since=`

`/stream/<patient>/<id>?since=<t>` returns only records with `t_end > t`. That is how a real
client follows a running refit: keep the last `t_end` you saw as a cursor and ask for
everything after it, instead of re-downloading the whole stream each poll.

One catch, and the notebook handles it: a **restarted** refit truncates the file and begins
again at `t_end = −60`, which is *behind* your cursor, so a `since=` poll would go silently
empty. `poll_new()` detects the shrink and rewinds.

In [ ]:
HISTORY = {p: [] for p in PARAMS}     # everything we have pulled, in arrival order
CURSOR  = {p: None for p in PARAMS}   # last t_end seen per parameter


def poll_new(patient=PATIENT, verbose=True):
    """Fetch only records newer than the cursor. Returns how many arrived."""
    got = 0
    on_disk = {k: v["n_records"] for k, v in api(f"/latest/{patient}")["params"].items()}
    for p in PARAMS:
        if on_disk.get(p, 0) < len(HISTORY[p]):            # refit restarted, file truncated
            if verbose:
                print(f"  {p}: stream restarted — rewinding cursor")
            HISTORY[p], CURSOR[p] = [], None

        q = f"/stream/{patient}/{p}"
        if CURSOR[p] is not None:
            q += f"?since={CURSOR[p]}"
        try:
            recs = api(q)["records"]
        except urllib.error.HTTPError:
            continue                                       # no stream for this param yet
        if recs:
            HISTORY[p].extend(recs)
            CURSOR[p] = recs[-1]["t_end"]
            got += len(recs)
            if verbose:
                r = recs[-1]
                print(f'  {p:<31} +{len(recs)} -> t_end {r["t_end"]:>7.1f}  '
                      f'{r["value"]:>10.4g}  {r["status"]}')
    return got


print("first poll (cursor empty, so this pulls everything):")
n = poll_new()
print(f"\n{n} records\n")

print("second poll (cursor now set — nothing new unless a refit wrote a window):")
n = poll_new()
print(f"\n{n} new records")

### Follow the stream

Run this while the refit is going. It polls until a target number of windows have arrived or
it times out, printing each record as it lands — this is the "realtime" view of the API.

In [ ]:
FOLLOW_SECONDS, POLL_EVERY, STOP_AFTER = 300, 5, 16   # timeout, poll interval, windows wanted

t0 = time.time()
while True:
    n = poll_new(verbose=False)
    have = len(HISTORY[PARAMS[0]])
    if n:
        r = HISTORY[PARAMS[0]][-1]
        print(f'[{time.strftime("%H:%M:%S")}] +{n} records, now {have} windows, '
              f't_end {r["t_end"]:.1f}s, rmse {r["rmse"]:.4g}')
    if not refit_alive():
        print("refit not running — nothing further will arrive")
        break
    if have >= STOP_AFTER or time.time() - t0 >= FOLLOW_SECONDS:
        break
    time.sleep(POLL_EVERY)

print(f'\ndone — {len(HISTORY[PARAMS[0]])} windows in hand, '
      f't_end up to {CURSOR[PARAMS[0]]}')

## 7. Watching the intervals collapse

This is the plot the API exists for. Each panel is one parameter against `t_end`:

- the shaded band is the 95% Fisher-information interval,
- the green strip is the healthy range from the contract,
- **hollow** markers are windows where `identifiable` is false — the point estimate is there,
  but it is a guess, not a measurement.

A band that runs off the top or bottom of a panel is clipped, not missing: some
pre-manoeuvre intervals span hundreds of decades, which is exactly the point.

Before the compression manoeuvre, resting ICP is nearly blind to the venous outflow path, so
the venous parameters start with intervals spanning orders of magnitude. The interesting
event is the moment the manoeuvre supplies the information and those intervals snap shut.
The y-axes are logarithmic because that collapse spans decades.

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False
    print("matplotlib not installed — `pip install matplotlib` for the plots.\n"
          "The text summary below works either way.")


def plot_streams(history=None, patient=PATIENT):
    h = history if history is not None else HISTORY
    have = [p for p in PARAMS if h.get(p)]
    if not have:
        print("no records to plot — run the refit (step 4) or poll (step 6) first")
        return
    if not HAVE_MPL:
        for p in have:
            print(f"\n{p}")
            for r in h[p]:
                ci = "—" if r["ci_lo"] is None else f'[{r["ci_lo"]:.3g}, {r["ci_hi"]:.3g}]'
                print(f'  t_end {r["t_end"]:>7.1f}  {r["value"]:>10.4g}  {ci:>26}  {r["status"]}')
        return

    fig, axes = plt.subplots(len(have), 1, figsize=(9, 2.5 * len(have)), sharex=True)
    axes = [axes] if len(have) == 1 else list(axes)
    for ax, p in zip(axes, have):
        recs = sorted(h[p], key=lambda r: r["t_end"])
        t  = [r["t_end"] for r in recs]
        v  = [r["value"] for r in recs]
        lo = [r["ci_lo"] if r["ci_lo"] is not None else float("nan") for r in recs]
        hi = [r["ci_hi"] if r["ci_hi"] is not None else float("nan") for r in recs]
        m  = meta[p]

        # An unidentifiable window can carry an interval spanning hundreds of decades
        # ([1e-267, 1e268] is real, in this data). Fit the axis to the estimates and the
        # bounds, and clip the band to the frame — a band running off the top or bottom of a
        # panel IS the "we do not know this yet" signal.
        finite = [x for x in v if x is not None and math.isfinite(x) and x > 0]
        ref    = [m["min_pathological_bound"], m["max_pathological_bound"]]
        ylo, yhi = min(finite + ref) / 30, max(finite + ref) * 30
        clip = lambda x: x if not math.isfinite(x) else min(max(x, ylo), yhi)
        lo, hi = [clip(x) for x in lo], [clip(x) for x in hi]

        ax.axhspan(m["min_healthy_bound"], m["max_healthy_bound"],
                   color="tab:green", alpha=0.12, lw=0, zorder=0, label="healthy band")
        ax.fill_between(t, lo, hi, color="tab:blue", alpha=0.20, lw=0,
                        label="95% interval")
        ax.plot(t, v, "-", color="tab:blue", lw=1.2, zorder=3)
        for r in recs:                                    # hollow == not identifiable
            ax.plot(r["t_end"], r["value"], "o", ms=5, zorder=4,
                    color="tab:blue" if r["identifiable"] else "none",
                    mec="tab:blue", mew=1.2)
        ax.set_yscale("log")
        ax.set_ylim(ylo, yhi)
        ax.set_ylabel(m["unit"], fontsize=8)
        ax.set_title(f'{p}   [{m["bound_provenance"]}]', fontsize=9, loc="left")
        ax.grid(alpha=0.25, which="both", lw=0.4)
    axes[-1].set_xlabel("t_end  (s) — end of the expanding window")
    axes[0].legend(fontsize=8, loc="upper right", ncol=2)
    fig.suptitle(f"patient {patient} — green = healthy band, hollow = unidentifiable",
                 fontsize=10, y=0.995)
    fig.tight_layout()
    plt.show()


plot_streams()

### The same thing in one number

Fractional interval width, `(ci_hi − ci_lo) / value`, on a log axis. The `identifiable` flag
is exactly `σ_fractional < 0.5`, so the dashed line is the threshold: everything above it is
a guess. This is the cleanest picture of information arriving.

In [ ]:
def plot_information(history=None):
    h = history if history is not None else HISTORY
    have = [p for p in PARAMS if h.get(p)]
    if not have:
        print("no records yet — run the refit (step 4) or poll (step 6) first")
        return
    if not HAVE_MPL:
        print("matplotlib not installed — this cell needs it")
        return
    WMAX = 1e6                      # widths run to 1e267 before the manoeuvre; clip them
    fig, ax = plt.subplots(figsize=(9, 4.5))
    seen = []
    for p in have:
        recs = sorted(h[p], key=lambda r: r["t_end"])
        t, w = [], []
        for r in recs:
            if r["ci_lo"] is not None and r["value"]:
                t.append(r["t_end"])
                w.append(min((r["ci_hi"] - r["ci_lo"]) / abs(r["value"]), WMAX))
        if t:
            seen += w
            ax.plot(t, w, "o-", ms=4, lw=1.2, label=p)
    thresh = math.exp(1.96 * 0.5) - math.exp(-1.96 * 0.5)   # the sigma < 0.5 cutoff, = 2.29
    if max(seen, default=0) >= WMAX:          # only if something actually hit the ceiling
        ax.axhline(WMAX, ls=":", c="0.55", lw=0.8)
        ax.text(ax.get_xlim()[0], WMAX, "  clipped — the real width runs to 1e267",
                va="top", fontsize=8, color="0.4")
    ax.axhline(thresh, ls="--", c="k", lw=0.8)
    ax.set_ylim(min(seen + [thresh]) / 3, max(seen + [thresh]) * 3)
    ax.text(ax.get_xlim()[0], thresh, "  identifiable below this line (sigma < 0.5)",
            va="bottom", fontsize=8)
    ax.set_yscale("log")
    ax.set_xlabel("t_end  (s)")
    ax.set_ylabel("fractional interval width  (ci_hi - ci_lo) / value")
    ax.set_title("information arriving: interval width per window", fontsize=10, loc="left")
    ax.grid(alpha=0.25, which="both", lw=0.4)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


plot_information()

### Live-updating plot

Re-polls and redraws on an interval. Run this alongside a real refit (step 4 with
`START_REFIT = True`) and the curves extend to the right as each window lands.

In [ ]:
LIVE_SECONDS, REDRAW_EVERY = 900, 15

t0 = time.time()
while True:
    poll_new(verbose=False)
    running = refit_alive()
    clear_output(wait=True)
    print(f'{time.strftime("%H:%M:%S")}   {len(HISTORY[PARAMS[0]])} windows   '
          f'refit: {"running" if running else "not running"}')
    plot_streams()
    if not running:                       # a static stream only needs drawing once
        break
    if time.time() - t0 >= LIVE_SECONDS:
        print(f"(stopped after {LIVE_SECONDS}s — re-run this cell to keep watching)")
        break
    time.sleep(REDRAW_EVERY)

## 8. Shut down

Stops the server and, if one is running, the refit. Skip the refit line if you want it to
carry on writing after the notebook is done — it is a plain background process, and because
the API is file-backed you can restart the server later and pick the stream up where it got
to.

In [ ]:
for name in ("REFIT", "SERVER"):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"{name} (pid {proc.pid}) stopped")
    else:
        print(f"{name}: not running")